# PRACTICE 2: PREDICTING PRODUCT SALES
## CHẶNG 3: HUẤN LUYỆN MÔ HÌNH, ĐÁNH GIÁ & GIẢI THÍCH (MODELING, EVALUATION & EXPLAINABILITY)

> **Mục tiêu kỹ thuật:**
> - Huấn luyện 3 mô hình hồi quy From Scratch tự viết bằng NumPy: Linear Regression, Decision Tree Regressor, Random Forest Regressor.
> - Kiểm định chéo 5-Fold Cross Validation trên tập Train để đo lường tính ổn định.
> - Đánh giá kiểm thử độc lập trên 20.571 mẫu Test set với phép nghịch đảo `np.expm1` (MSE, RMSE INR, MAE INR, R²).
> - Kỹ thuật Cô Tiên: Bóc tách Top 15 đặc trưng quan trọng nhất (Feature Importances) và xuất `best_model.joblib`, `metadata.json`.

### 1. Khởi tạo 3 Mô hình Scratch & Đánh giá chéo 5-Fold Cross Validation — Võ Khôi Nguyên
> **Nhiệm vụ Võ Khôi Nguyên:** Hoàn thiện module `scripts/model_training.py` (`create_scratch_models`, `build_training_pipelines`, `evaluate_cv_models`, `train_and_save_all_models`).
- 3 mô hình: Linear Regression (OLS Normal Equation), Decision Tree Regressor (Quantile Splits), Random Forest Regressor (Ensemble Bagging).
- Thực hiện 5-Fold Cross Validation trên tập Train để đo lường phương sai và độ lệch chuẩn.

In [ ]:
# TODO: [Võ Khôi Nguyên] Đóng gói Pipeline hồi quy và thực hiện kiểm định chéo 5-Fold Cross Validation
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from scripts.model_training import (
    create_scratch_models,
    build_training_pipelines,
    evaluate_cv_models,
    train_and_save_all_models
)
from scripts.model_evaluation import (
    evaluate_holdout_test,
    extract_feature_importances,
    export_evaluation_artifacts
)

data_path = BASE_DIR / 'data' / 'processed' / 'train_test_data.joblib'
prep_path = BASE_DIR / 'models' / 'saved_models' / 'preprocessor.joblib'

data = joblib.load(data_path)
preprocessor = joblib.load(prep_path)
X_train, y_train = data['X_train'], data['y_train']
X_test, y_test = data['X_test'], data['y_test']
y_test_raw = data['y_test_raw']

pipelines = build_training_pipelines(preprocessor)
print("✓ Bắt đầu 5-Fold Cross Validation trên tập Train...")
cv_results_df = evaluate_cv_models(pipelines, X_train, y_train, cv=5, sample_size=10000)
display(cv_results_df)

In [ ]:
# TODO: [Võ Khôi Nguyên] Huấn luyện toàn bộ các Pipeline trên 82.281 mẫu Train và lưu mô hình tối ưu best_model.joblib
# Huấn luyện cả 3 Pipeline trên toàn bộ 82,281 mẫu Train
saved_dir = BASE_DIR / 'models' / 'saved_models'
trained_pipelines = train_and_save_all_models(pipelines, X_train, y_train, saved_dir)
print("✓ Hoàn tất huấn luyện toàn bộ các mô hình hồi quy trên tập Train.")

# TODO: [Mai Hoàng Danh] Trực quan hóa Biểu đồ Thực tế vs Dự đoán & Phân tích Phần dư Residual Plot
# Triển khai và gọi các hàm plot_actual_vs_predicted và plot_residuals từ scripts/model_evaluation.py
from scripts.model_evaluation import plot_actual_vs_predicted, plot_residuals

best_pipe = trained_pipelines['Decision Tree Regressor']
y_pred_log = best_pipe.predict(X_test)
y_pred_inr = np.clip(np.expm1(y_pred_log), 0.0, None)

plot_actual_vs_predicted(y_test_raw, y_pred_inr, max_points=1000)
plot_residuals(y_test_raw, y_pred_inr, max_points=1000)


In [ ]:
# TODO: [Mai Hoàng Danh] Đánh giá độc lập trên 20.571 mẫu Test set và nghịch đảo expm1 về giá trị tiền tệ thực tế INR
# Đánh giá kiểm thử trên tập Test
test_metrics_df = evaluate_holdout_test(trained_pipelines, X_test, y_test, y_test_raw)
print("=== BẢNG TỔNG KẾT HIỆU NĂNG MÔ HÌNH TRÊN TẬP TEST ĐỘC LẬP (20.571 MẪU) ===")
display(test_metrics_df)

In [ ]:
# TODO: [Mai Hoàng Danh] Trực quan hóa Biểu đồ Thực tế vs Dự đoán & Phân tích Phần dư Residual Plot
# Trực quan hóa Thực tế vs Dự đoán & Phân tích Phần dư (Residual Plot)
best_pipe = trained_pipelines['Decision Tree Regressor']
y_pred_log = best_pipe.predict(X_test)
y_pred_inr = np.clip(np.expm1(y_pred_log), 0.0, None)
residuals = y_test_raw - y_pred_inr

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Thực tế vs Dự đoán
axes[0].scatter(y_test_raw[:1000], y_pred_inr[:1000], alpha=0.3, color='#2980b9', s=15)
max_val = max(y_test_raw[:1000].max(), y_pred_inr[:1000].max())
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=2, label='Đường lý tưởng y = ŷ')
axes[0].set_title('Thực tế vs Dự đoán (1.000 mẫu Test)', fontweight='bold')
axes[0].set_xlabel('Doanh số thực tế (INR)')
axes[0].set_ylabel('Doanh số dự đoán (INR)')
axes[0].legend()

# 2. Biểu đồ phần dư
axes[1].scatter(y_pred_inr[:1000], residuals[:1000], alpha=0.3, color='#e67e22', s=15)
axes[1].axhline(0, color='black', linestyle='--', lw=1.5)
axes[1].set_title('Biểu đồ phần dư Residual Plot (ŷ vs e)', fontweight='bold')
axes[1].set_xlabel('Doanh số dự đoán (INR)')
axes[1].set_ylabel('Phần dư (y - ŷ)')

plt.tight_layout()
plt.show()
print(f"• Phần dư trung bình (Mean Residual): {residuals.mean():.2f} INR (Gần 0: Không thiên lệch)")

In [ ]:
# TODO: [Mai Hoàng Danh] Trích xuất Top 15 đặc trưng quan trọng nhất (Feature Importances) và xuất metadata.json
# Kỹ thuật Cô Tiên: Trích xuất và trực quan hóa Top 15 đặc trưng quan trọng
top15_features = extract_feature_importances(best_pipe, top_n=15)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=top15_features, palette='mako')
plt.title('Top 15 Đặc trưng quan trọng nhất (Feature Importances)', fontweight='bold')
plt.xlabel('Tỷ lệ đóng góp giảm phương sai')
plt.tight_layout()
plt.show()
display(top15_features)

In [ ]:
# Xuất siêu dữ liệu nghiệm thu metadata.json
meta_path = export_evaluation_artifacts(
    model_name='Decision Tree Regressor (Optimized Scratch)',
    metrics_dict=test_metrics_df.iloc[1].to_dict(),
    top_features=top15_features,
    output_dir=saved_dir
)
print(f"✓ Đã xuất siêu dữ liệu nghiệm thu thành công tại: {meta_path}")